# Saludos de Forty5 con la voz de Epachuko (Kaggle)

Graba todas las frases de `shared/voz/saludos.json` y sube cada MP3 al repositorio de Sportfy.
Settings: **Accelerator → GPU T4 x2**, **Internet → On**. Add-ons → Secrets: **GH_TOKEN** marcado.
Lanzar con **Save Version → Save & Run All**. Si se corta, al lanzarlo otra vez sigue por donde iba.

In [ ]:
# 1. GPU y clave de GitHub
import os, subprocess
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("❌ No hay GPU. En Settings → Accelerator elige GPU T4 x2.")
print("✅", gpu.splitlines()[0])
from kaggle_secrets import UserSecretsClient
try:
    os.environ["GH_TOKEN"] = UserSecretsClient().get_secret("GH_TOKEN")
    print("🔑 Clave de GitHub lista")
except Exception:
    raise SystemExit("❌ No encuentro el secreto GH_TOKEN. Añádelo en Add-ons → Secrets y márcalo para este cuaderno.")

In [ ]:
# 2. Instalar el estudio y el motor de voz
if os.path.isdir("/kaggle/working/estudio-de-voz/.git"):
    subprocess.run(["git", "-C", "/kaggle/working/estudio-de-voz", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/Jonragitt/estudio-de-voz", "/kaggle/working/estudio-de-voz"], check=True)
r = subprocess.run("pip install -q chatterbox-tts yt-dlp", shell=True, capture_output=True, text=True)
subprocess.run("pip uninstall -y -q torchvision", shell=True)
r2 = subprocess.run(["python", "-c", "from transformers import LlamaModel; import chatterbox.mtl_tts"], capture_output=True, text=True)
if r2.returncode != 0:
    print(r.stdout[-2000:], r.stderr[-2000:], r2.stderr[-3000:])
    raise SystemExit("❌ El motor de voz no se ha instalado bien (mira el error de arriba).")
print("✅ Motor de voz instalado")

In [ ]:
# 3. Grabar y subir a GitHub (todas las frases; MITAD="par"/"impar" para repartir con Colab)
import sys, json, urllib.request
RAMA = "claude/project-thread-iw4b9m"
MITAD = ""
frases = json.load(urllib.request.urlopen(f"https://raw.githubusercontent.com/Jonragitt/Sportfy/{RAMA}/shared/voz/saludos.json"))["lecciones"]
ids = [l["id"] for i, l in enumerate(frases) if not MITAD or i % 2 == (0 if MITAD == "par" else 1)]
print(f"🎙️ Kaggle: {len(ids)} de {len(frases)} frases")
env = dict(os.environ, REPO="jonragitt/sportfy", RAMA=RAMA, TEXTOS="shared/voz/saludos.json", SALIDA="shared/voz",
           VOZ="shared/voz/ref-kaggle", SOLO=",".join(ids), EXPRESIVIDAD="0.5", PAUSAS="0.6")
p = subprocess.Popen([sys.executable, "-u", "kaggle/narrar_repo.py"], cwd="/kaggle/working/estudio-de-voz",
                     env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for linea in p.stdout:
    print(linea, end="")
p.wait()
if p.returncode:
    raise SystemExit(f"❌ La narración se ha parado (código {p.returncode}). Vuelve a lanzarlo: sigue por donde iba.")